In [52]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [53]:
df = pd.read_csv('./dataset/dataset.csv')

In [54]:
df = df.drop(columns=['id',  'Work Pressure', 'Job Satisfaction'])

In [55]:
for col in ['City', 'Degree', 'Profession']:
    counts = df[col].value_counts()
    df[col] = df[col].where(df[col].isin(counts[counts >= 10].index), other=pd.NA)

In [56]:
print(df.head())

   Gender   Age           City Profession  Academic Pressure  CGPA  \
0    Male  33.0  Visakhapatnam    Student                5.0  8.97   
1  Female  24.0      Bangalore    Student                2.0  5.90   
2    Male  31.0       Srinagar    Student                3.0  7.03   
3  Female  28.0       Varanasi    Student                3.0  5.59   
4  Female  25.0         Jaipur    Student                4.0  8.13   

   Study Satisfaction     Sleep Duration Dietary Habits   Degree  \
0                 2.0          5-6 hours        Healthy  B.Pharm   
1                 5.0          5-6 hours       Moderate      BSc   
2                 5.0  Less than 5 hours        Healthy       BA   
3                 2.0          7-8 hours       Moderate      BCA   
4                 3.0          5-6 hours       Moderate   M.Tech   

  Have you ever had suicidal thoughts ?  Work/Study Hours  Financial Stress  \
0                                   Yes               3.0               1.0   
1           

In [57]:
df = df.dropna()
df = df.drop_duplicates()

In [58]:
X = df.drop(columns=['Depression'])
y = df['Depression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

Train: 22272 | Test: 5569


In [59]:
numerical_cols   = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'str']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (22272, 72) | X_test: (5569, 72)


In [60]:
all_cols = numerical_cols + preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()

X_train_dense = X_train.toarray() if hasattr(X_train, 'toarray') else X_train

cleaned_df = pd.DataFrame(X_train_dense, columns=all_cols)
cleaned_df['Depression'] = y_train.values

cleaned_df.to_csv('./dataset/cleaned_dataset.csv', index=False)